# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!git clone https://github.com/MarriamFatima-alt/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 122, done.
remote: Counting objects: 100% (122/122), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 122 (delta 39), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (122/122), 1.85 MiB | 10.32 MiB/s, done.
Resolving deltas: 100% (39/39), done.


In [3]:
%cd flyrank-ml-internship
!python scripts/01_prepare_features.py
!python scripts/02_baseline_score.py
!python scripts/03_train_model.py
!python scripts/04_evaluate_and_export.py

/content/flyrank-ml-internship
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv
Wrote baseline queue: /content/flyrank-ml-internship/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-ml-internship/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-ml-internship/outputs/model_results.json
Wrote final refresh queue: /content/flyrank-ml-internship/outputs/refresh_queue.csv
Wrote model report: /content/flyrank-ml-internship/outputs/model_report.md
Wrote charts in: /content/flyrank-ml-internship/outputs/charts


In [4]:
!cat outputs/model_results.json

{
  "baseline": {
    "baseline_accuracy": 0.6086021505376344,
    "baseline_average_precision": 0.46760658636934826,
    "baseline_f1": 0.2743221690590112,
    "baseline_precision": 0.4985507246376812,
    "baseline_precision_at_100": 0.36,
    "baseline_precision_at_20": 0.15,
    "baseline_precision_at_50": 0.24,
    "baseline_recall": 0.18921892189218922,
    "baseline_roc_auc": 0.6268917852237201
  },
  "best_model": {
    "feature_importance_top": [
      {
        "feature": "days_with_impressions",
        "importance": 0.15814381552310994
      },
      {
        "feature": "log_impressions_90d",
        "importance": 0.12863800983285106
      },
      {
        "feature": "avg_position",
        "importance": 0.10916404306325882
      },
      {
        "feature": "content_age_days",
        "importance": 0.0951682857931063
      },
      {
        "feature": "char_count",
        "importance": 0.042608309447693275
      },
      {
        "feature": "word_count",
        "im

In [5]:
import json
with open("outputs/model_results.json") as f:
    data = json.load(f)
print(json.dumps(data["models"], indent=2))

{
  "decision_tree": {
    "accuracy": 0.6765591397849462,
    "average_precision": 0.5753189989001579,
    "f1": 0.6338851022395326,
    "precision": 0.5685589519650655,
    "precision_at_100": 0.62,
    "precision_at_20": 0.45,
    "precision_at_50": 0.58,
    "recall": 0.7161716171617162,
    "roc_auc": 0.7415203737887913
  },
  "logistic_regression": {
    "accuracy": 0.6606451612903226,
    "average_precision": 0.5215418768632023,
    "f1": 0.5662451896646509,
    "precision": 0.5659340659340659,
    "precision_at_100": 0.44,
    "precision_at_20": 0.35,
    "precision_at_50": 0.4,
    "recall": 0.5665566556655666,
    "roc_auc": 0.7002914980763613
  },
  "random_forest": {
    "accuracy": 0.672258064516129,
    "average_precision": 0.6182189675945406,
    "f1": 0.6395458845789972,
    "precision": 0.5609958506224066,
    "precision_at_100": 0.72,
    "precision_at_20": 0.65,
    "precision_at_50": 0.74,
    "recall": 0.7436743674367436,
    "roc_auc": 0.7500295227262839
  }
}


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [6]:
# Method:  models (Logistic Regression, Decision Tree, Random Forest)
# reference pipeline (scripts/03_train_model.py)
# Random Forest best model  based on Precision@50.
print("Compared: logistic_regression, decision_tree, random_forest")
print("Best model: random_forest (highest precision_at_50)")

Compared: logistic_regression, decision_tree, random_forest
Best model: random_forest (highest precision_at_50)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [7]:
print("Split strategy: client_holdout")

Split strategy: client_holdout


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [8]:
import pandas as pd

comparison = pd.DataFrame({
    "Precision@20": [0.15, 0.35, 0.45, 0.65],
    "Precision@50": [0.24, 0.40, 0.58, 0.74],
    "Precision@100": [0.36, 0.44, 0.62, 0.72],
    "ROC-AUC": [0.627, 0.700, 0.742, 0.750],
    "Avg Precision": [0.468, 0.522, 0.575, 0.618],
}, index=["Baseline (rule)", "Logistic Regression", "Decision Tree", "Random Forest"])

comparison

,Precision@20,Precision@50,Precision@100,ROC-AUC,Avg Precision
Baseline (rule),0.15,0.24,0.36,0.627,0.468
Logistic Regression,0.35,0.40,0.44,0.700,0.522
Decision Tree,0.45,0.58,0.62,0.742,0.575
Random Forest,0.65,0.74,0.72,0.750,0.618


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [9]:
print("Top features driving predictions:")
print("1. days_with_impressions (0.158)")
print("2. log_impressions_90d (0.129)")
print("3. avg_position (0.109)")
print("4. content_age_days (0.095)")
print("5. char_count (0.043)")

Top features driving predictions:
1. days_with_impressions (0.158)
2. log_impressions_90d (0.129)
3. avg_position (0.109)
4. content_age_days (0.095)
5. char_count (0.043)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.